# 🤖 Aula 7 — Métodos e estratégias de busca (UCS e A*)
# 🧹 Busca no Vacuum World

**Continuação da aula anterior:** agora vamos comparar estratégias que consideram o **custo do caminho**.

### Aula 6
🔍 BFS e DFS → diferentes ordens de exploração

### Aula 7
💰 UCS → menor custo acumulado
🧭 A* → custo acumulado + heurística

**Pergunta central:** como escolher o próximo estado quando o custo importa?


## 🎯 Objetivos

Ao final desta prática, você deverá ser capaz de:

- compreender como o **UCS** escolhe estados pela menor custo acumulado;
- compreender como o **A*** combina custo acumulado e heurística;
- executar os dois algoritmos no **Vacuum World com 5 salas**;
- comparar custo e estados expandidos;
- observar como a heurística influencia a busca.


# 🔒 Infraestrutura

O ambiente já está pronto.

Hoje, o foco é observar e experimentar **UCS e A***.

Ambiente:

```text
┌──────────┬──────────┬──────────┬──────────┬──────────┐
│    A     │    B     │    C     │    D     │    E     │
└──────────┴──────────┴──────────┴──────────┴──────────┘
```

Cada sala pode estar `Clean` ou `Dirty`.

Ações: `Suck`, `Left`, `Right`.

O estado é representado por:

```text
(posição, condição_A, condição_B, condição_C, condição_D, condição_E)
```


In [ ]:
from typing import Tuple, Optional
from heapq import heappush, heappop
from itertools import product
import random

# ============================================================
# REPRESENTAÇÃO DO VACUUM WORLD
# ============================================================

Position = str
Condition = str
State = Tuple[Position, Condition, Condition, Condition, Condition, Condition]

POSITIONS = ["A", "B", "C", "D", "E"]
ACTIONS = ["Suck", "Left", "Right"]


In [ ]:
# ============================================================
# ESTADO INICIAL
# ============================================================

# A 🦠 ↔ B ✨ ↔ C 🤖🦠 ↔ D ✨ ↔ E 🦠
INITIAL_STATE: State = (
      "C",      # posição do agente
      "Dirty",  # condição da sala A
      "Clean",  # condição da sala B
      "Dirty",  # condição da sala C
      "Clean",  # condição da sala D
      "Dirty"   # condição da sala E
)


In [ ]:
# ============================================================
# VISUALIZAÇÃO
# ============================================================

def describe_state(state: State):
    # O primeiro elemento informa a posição do agente.
    position = state[0]

    # Os demais elementos representam as salas.
    rooms = state[1:]

    parts = []

    # Percorremos simultaneamente posições e condições.
    for room, condition in zip(POSITIONS, rooms):
        icon = "🦠" if condition == "Dirty" else "✨"
        robot = "🤖" if room == position else ""
        parts.append(f"{room} {robot}{icon}")

    print("  ↔  ".join(parts))

describe_state(INITIAL_STATE)


A 🦠  ↔  B ✨  ↔  C 🤖🦠  ↔  D ✨  ↔  E 🦠


In [ ]:
# ============================================================
# 🎯 TESTE DE OBJETIVO
# ============================================================

def is_goal(state: State) -> bool:
    # A posição final do agente não importa.
    # Um objetivo é alcançado quando todas as salas estão limpas.
    rooms = state[1:]
    return all(room == "Clean" for room in rooms)


In [ ]:
# ============================================================
# 🔄 TRANSIÇÃO DE ESTADOS
# ============================================================

def apply_action(state: State, action: str) -> Optional[State]:
    """Aplica uma ação e retorna um NOVO estado.

    Se a ação for inválida ou não gerar um novo estado,
    retorna None.
    """

    position = state[0]

    # Lista temporária para permitir alteração da sala atual.
    rooms = list(state[1:])

    # Índice da posição atual em A, B, C, D, E.
    index = POSITIONS.index(position)

    if action == "Suck":
        if rooms[index] == "Dirty":
            rooms[index] = "Clean"
            return (position, *rooms)
        return None

    elif action == "Left":
        if index > 0:
            new_position = POSITIONS[index - 1]
            return (new_position, *rooms)
        return None

    elif action == "Right":
        if index < len(POSITIONS) - 1:
            new_position = POSITIONS[index + 1]
            return (new_position, *rooms)
        return None

    return None


In [ ]:
# ============================================================
# 🌳 GERANDO SUCESSORES
# ============================================================

def successors(state: State):
    """Retorna pares (ação, novo_estado)."""

    result = []

    # Para cada ação possível, tentamos gerar um novo estado.
    for action in ACTIONS:
        new_state = apply_action(state, action)

        # Apenas transições válidas são mantidas.
        if new_state is not None:
            result.append((action, new_state))

    return result


In [ ]:
# ============================================================
# EXPERIMENTO — SUCESSORES
# ============================================================

print("Estado atual:")
describe_state(INITIAL_STATE)

print("\nEstados sucessores:")

for action, new_state in successors(INITIAL_STATE):
    print(f"\nAção: {action}")
    describe_state(new_state)


Estado atual:
A 🦠  ↔  B ✨  ↔  C 🤖🦠  ↔  D ✨  ↔  E 🦠

Estados sucessores:

Ação: Suck
A 🦠  ↔  B ✨  ↔  C 🤖✨  ↔  D ✨  ↔  E 🦠

Ação: Left
A 🦠  ↔  B 🤖✨  ↔  C 🦠  ↔  D ✨  ↔  E 🦠

Ação: Right
A 🦠  ↔  B ✨  ↔  C 🦠  ↔  D 🤖✨  ↔  E 🦠


# 🌐 A fronteira

Os estados descobertos, mas ainda não explorados, formam a:

# **FRONTEIRA**

Depois de gerar sucessores, surge a pergunta:

> **Qual estado devemos explorar em seguida?**

A resposta depende da estratégia de busca.


# 🔒 Reconstrução do plano

Quando encontramos o objetivo, precisamos recuperar a sequência de ações utilizada.

Esta função já está pronta. Vamos utilizá-la durante a busca.


In [ ]:
# ============================================================
# 🔒 RECONSTRUÇÃO DO PLANO
# ============================================================

def reconstruct_path(parent, goal_state):
    plan = [] # Lista onde vamos guardar as ações
    current = goal_state # Começamos pelo objetivo

    # Enquanto não chegarmos ao estado inicial
    while parent[current][0] is not None:

        # Descobrimos:
        # de qual estado viemos, qual ação usamos
        previous_state, action = parent[current]
        # Guardamos a ação
        plan.append(action)
        # Voltamos para o estado anterior
        current = previous_state

    # Como recuperamos o caminho de trás para frente,
    # precisamos inverter as ações.
    plan.reverse()
    return plan


# 🟦 Uniform Cost Search (UCS)

O **UCS** escolhe, na fronteira, o estado com o menor **custo acumulado**.

```text
UCS → menor g(n)
g(n) = custo do caminho desde o estado inicial até n
```

**Não importa quantos passos foram necessários. O que importa é quanto o caminho custou.**

No exemplo atual, todas as ações custam 1. Depois vamos alterar os custos para perceber a diferença.


In [ ]:
# ============================================================
# UCS — CÓDIGO PRONTO
# ============================================================


def ucs(initial_state: State):

    # --------------------------------------------------------
    # 1. FRONTEIRA
    # --------------------------------------------------------
    # A fronteira guarda os estados que ainda precisam ser
    # explorados.
    #
    # No UCS, a fronteira funciona como uma FILA DE PRIORIDADE:
    # o próximo estado escolhido será sempre aquele cujo
    # caminho possui o MENOR CUSTO ACUMULADO.
    #
    # Exemplo:
    #   custo 2 → prioridade maior
    #   custo 5 → prioridade menor
    #
    # Portanto, o UCS não escolhe simplesmente o estado mais
    # "perto". Ele escolhe o caminho de menor custo.
    frontier = []

    # Contador usado apenas para desempatar estados que possuem
    # o mesmo custo.
    #
    # Ele NÃO faz parte da lógica do UCS.
    counter = 0

    # Inserimos o estado inicial na fronteira.
    #
    # A tupla possui:
    #   (custo, contador, estado, plano)
    #
    # custo = 0
    #   Ainda não realizamos nenhuma ação.
    #
    # contador = 0
    #   Usado apenas para desempate.
    #
    # initial_state
    #   Estado inicial do problema.
    #
    # []
    #   Nenhuma ação foi realizada até agora.
    heappush(frontier, (0, counter, initial_state, []))


    # --------------------------------------------------------
    # 2. MELHOR CUSTO CONHECIDO
    # --------------------------------------------------------
    # Guarda o menor custo encontrado até agora para chegar
    # a cada estado.
    #
    # Exemplo:
    #
    #   estado A → custo 0
    #   estado B → custo 3
    #   estado C → custo 5
    #
    # Se encontrarmos posteriormente um caminho para C com
    # custo 2, atualizamos:
    #
    #   C → 2
    #
    # Isso evita continuar usando caminhos mais caros.
    best_cost = {initial_state: 0}


    # Quantidade de estados que já foram efetivamente
    # expandidos durante a busca.
    #
    # É uma métrica útil para analisar a eficiência do algoritmo.
    expanded = 0


    # --------------------------------------------------------
    # 3. BUSCA
    # --------------------------------------------------------
    # Enquanto houver estados aguardando para serem explorados...
    while frontier:

        # Retira da fronteira o estado de MENOR CUSTO.
        #
        # Retornamos:
        #   cost         → custo acumulado do caminho
        #   _            → contador (não precisamos dele aqui)
        #   current_state → estado que será analisado
        #   plan         → sequência de ações realizadas
        cost, _, current_state, plan = heappop(frontier)


        # ----------------------------------------------------
        # 4. VERIFICAÇÃO DE ENTRADA DESATUALIZADA
        # ----------------------------------------------------
        # Um mesmo estado pode ter sido colocado na fronteira
        # mais de uma vez.
        #
        # Exemplo:
        #   encontramos C com custo 8
        #   depois encontramos C com custo 5
        #
        # A entrada de custo 8 continua na fila, mas deixou de
        # ser a melhor opção.
        #
        # Se o custo retirado da fila não for mais o melhor custo
        # conhecido para aquele estado, simplesmente ignoramos
        # essa entrada.
        if cost != best_cost.get(current_state):
            continue


        # ----------------------------------------------------
        # 5. TESTE DE OBJETIVO
        # ----------------------------------------------------
        # Verificamos se o estado atual já resolve o problema.
        #
        # No Vacuum World, por exemplo:
        #   objetivo = todas as salas limpas.
        #
        # Se chegamos ao objetivo, não precisamos continuar
        # procurando.
        if is_goal(current_state):
            return plan, cost, expanded


        # O estado será efetivamente explorado agora.
        expanded += 1


        # ----------------------------------------------------
        # 6. GERAÇÃO DOS SUCESSORES
        # ----------------------------------------------------
        # A partir do estado atual, descobrimos quais ações
        # podem ser realizadas.
        #
        # Cada sucessor representa:
        #   ação realizada + novo estado resultante
        #
        # Exemplo no Vacuum World:
        #   RIGHT → novo estado
        #   SUCK  → novo estado
        #   LEFT  → novo estado
        for action, new_state in successors(current_state):


            # ------------------------------------------------
            # 7. CUSTO DA AÇÃO
            # ------------------------------------------------
            # Nesta versão didática, todas as ações possuem
            # custo 1.
            #
            # Portanto:
            #
            #   mover = 1
            #   aspirar = 1
            #
            # O custo do novo caminho será o custo que já temos
            # + o custo da nova ação.
            action_cost = 1

            new_cost = cost + action_cost


            # ------------------------------------------------
            # 8. VERIFICAR SE ENCONTRAMOS UM CAMINHO MELHOR
            # ------------------------------------------------
            # Só adicionamos o novo estado à fronteira se o
            # caminho encontrado for MAIS BARATO que o melhor
            # caminho conhecido anteriormente.
            #
            # Se ainda não conhecemos esse estado, usamos
            # infinito como custo inicial.
            if new_cost < best_cost.get(new_state, float("inf")):

                # Registramos o novo melhor custo.
                best_cost[new_state] = new_cost

                # Atualizamos o contador para desempate.
                counter += 1

                # Colocamos o novo estado na fila de prioridade.
                #
                # A prioridade será o new_cost.
                #
                # Também guardamos:
                #   new_state → estado alcançado
                #   plan + [action] → plano atualizado
                heappush(
                    frontier,
                    (new_cost, counter, new_state, plan + [action])
                )


    # --------------------------------------------------------
    # 9. SEM SOLUÇÃO
    # --------------------------------------------------------
    # Se a fronteira ficar vazia, significa que todos os estados
    # possíveis foram explorados e nenhum atingiu o objetivo.
    return None, None, expanded


In [ ]:
# 🧪 TESTE — UCS

ucs_plan, ucs_cost, ucs_expanded = ucs(INITIAL_STATE)

print("UCS")
print("Plano:", ucs_plan)
print("Custo:", ucs_cost)
print("Estados expandidos:", ucs_expanded)


UCS
Plano: ['Suck', 'Left', 'Left', 'Suck', 'Right', 'Right', 'Right', 'Right', 'Suck']
Custo: 9
Estados expandidos: 32


# 🟪 A*

O **A*** usa o custo acumulado e uma estimativa do custo restante.

\[f(n) = g(n) + h(n)\]

- `g(n)` → custo já realizado
- `h(n)` → estimativa do custo restante
- `f(n)` → prioridade do estado


Para este Vacuum World simples, vamos usar:

```text
f(n) = g(n) + h(n)
g(n) = custo do caminho desde o estado inicial até n
h(n) = número de salas ainda sujas
```


In [ ]:
# ============================================================
# HEURÍSTICA
# ============================================================


def heuristic(state: State):
    # Estimativa simples: cada sala suja exige pelo menos uma ação SUCK.
    return sum(condition == "Dirty" for condition in state[1:])

print("Heurística do estado inicial:", heuristic(INITIAL_STATE))


Heurística do estado inicial: 3


In [ ]:
# ============================================================
# A* — CÓDIGO PRONTO
# ============================================================


def a_star(initial_state: State):
    frontier = []
    counter = 0

    g_initial = 0
    f_initial = g_initial + heuristic(initial_state)

    # (f(n), contador, g(n), estado, plano)
    heappush(
        frontier,
        (f_initial, counter, g_initial, initial_state, [])
    )

    best_g = {initial_state: 0}
    expanded = 0

    while frontier:
        f, _, g, current_state, plan = heappop(frontier)

        if g != best_g.get(current_state):
            continue

        if is_goal(current_state):
            return plan, g, expanded

        expanded += 1

        for action, new_state in successors(current_state):
            action_cost = 1
            new_g = g + action_cost

            if new_g < best_g.get(new_state, float("inf")):
                best_g[new_state] = new_g
                new_f = new_g + heuristic(new_state)
                counter += 1

                heappush(
                    frontier,
                    (new_f, counter, new_g, new_state, plan + [action])
                )

    return None, None, expanded


In [ ]:
# 🧪 TESTE — A*

astar_plan, astar_cost, astar_expanded = a_star(INITIAL_STATE)

print("A*")
print("Plano:", astar_plan)
print("Custo:", astar_cost)
print("Estados expandidos:", astar_expanded)


A*
Plano: ['Suck', 'Left', 'Left', 'Suck', 'Right', 'Right', 'Right', 'Right', 'Suck']
Custo: 9
Estados expandidos: 32


# 💰 Experimento — e se os custos forem diferentes?

Até aqui, todas as ações custam 1. Nesse caso, o UCS se comporta de forma muito parecida com o BFS.

No Challenge 5×5 teremos **custos diferentes por sala visitada**.

Vamos simular isso alterando a função de custo das ações.


In [ ]:
# ============================================================
# UCS — COM CUSTOS DIFERENTES
# ============================================================

# Versão simples: movimento continua custando 1, mas SUCK custa 2.
# Observe como o UCS continua escolhendo pelo custo acumulado.

ACTION_COST = {
    "Suck": 2,
    "Left": 1,
    "Right": 1
}


def ucs_with_costs(initial_state: State):
    frontier = []
    counter = 0
    heappush(frontier, (0, counter, initial_state, []))
    best_cost = {initial_state: 0}
    expanded = 0

    while frontier:
        cost, _, state, plan = heappop(frontier)

        if cost != best_cost.get(state):
            continue

        if is_goal(state):
            return plan, cost, expanded

        expanded += 1

        for action, new_state in successors(state):
            new_cost = cost + ACTION_COST[action]

            if new_cost < best_cost.get(new_state, float("inf")):
                best_cost[new_state] = new_cost
                counter += 1
                heappush(frontier, (new_cost, counter, new_state, plan + [action]))

    return None, None, expanded


In [ ]:
cost_plan, cost_total, cost_expanded = ucs_with_costs(INITIAL_STATE)

print("UCS com custos diferentes")
print("Plano:", cost_plan)
print("Custo total:", cost_total)
print("Estados expandidos:", cost_expanded)


UCS com custos diferentes
Plano: ['Left', 'Left', 'Suck', 'Right', 'Right', 'Suck', 'Right', 'Right', 'Suck']
Custo total: 12
Estados expandidos: 34


In [ ]:
# ============================================================
# A* — COM CUSTOS DIFERENTES
# ============================================================

ACTION_COST = {
    "Suck": 2,
    "Left": 1,
    "Right": 1
}


def heuristic2(state: State):

    position, dirty_rooms = state

    # Se não há salas sujas, já estamos no objetivo.
    if not dirty_rooms:
        return 0

    # Distância até a sala suja mais próxima.
    distance_to_dirty = min(
        abs(position - room)
        for room in dirty_rooms
    )

    # Cada sala suja ainda precisa ser aspirada.
    # Como Suck custa 2, temos pelo menos:
    # 2 × número de salas sujas
    cleaning_cost = 2 * len(dirty_rooms)

    # Estimativa do custo restante.
    return distance_to_dirty + cleaning_cost

def astar_with_costs(initial_state: State):
    frontier = []
    counter = 0
    initial_h = heuristic(initial_state)
    heappush(frontier,(initial_h, counter, 0, initial_state, []))
    best_cost = {initial_state: 0}
    expanded = 0

    while frontier:
        f_cost, _, cost, current_state, plan = heappop(frontier)

        if cost != best_cost.get(current_state):
            continue

        if is_goal(current_state):
            return plan, cost, expanded

        expanded += 1

        for action, new_state in successors(current_state):
            action_cost = ACTION_COST[action]
            new_cost = cost + action_cost

            if new_cost < best_cost.get(new_state, float("inf")):
                best_cost[new_state] = new_cost
                counter += 1

                h_cost = heuristic(new_state)
                f_cost = new_cost + h_cost

                heappush(frontier,(f_cost,counter,new_cost,new_state,plan + [action]))

    return None, None, expanded

In [ ]:
cost_plan, cost_total, cost_expanded = ucs_with_costs(INITIAL_STATE)

print("UCS com custos diferentes")
print("Plano:", cost_plan)
print("Custo total:", cost_total)
print("Estados expandidos:", cost_expanded)

cost_plan, cost_total, cost_expanded = astar_with_costs(INITIAL_STATE)

print("A* com custos diferentes")
print("Plano:", cost_plan)
print("Custo total:", cost_total)
print("Estados expandidos:", cost_expanded)

UCS com custos diferentes
Plano: ['Left', 'Left', 'Suck', 'Right', 'Right', 'Suck', 'Right', 'Right', 'Suck']
Custo total: 12
Estados expandidos: 34
A* com custos diferentes
Plano: ['Suck', 'Left', 'Left', 'Suck', 'Right', 'Right', 'Right', 'Right', 'Suck']
Custo total: 12
Estados expandidos: 32


# 🧠 Síntese

**Estado atual**
↓
**Gerar sucessores**
↓
**Fronteira**
↓
**Estratégia de busca**
↓
**Exploração**
↓
**Solução**

### UCS
💰 escolhe o menor **g(n)**

### A*
🧭 escolhe o menor **f(n) = g(n) + h(n)**

> O problema é o mesmo. A estratégia de busca muda a ordem de exploração.


# 🔜 Próxima etapa — Vacuum Search Challenge

Agora vamos aumentar a complexidade do Vacuum World:

```text
5 × 5
+ obstáculos
+ custos diferentes
+ várias salas sujas
```

No Challenge, vocês deverão **implementar um planejador** capaz de encontrar uma sequência de ações para limpar o ambiente.

As estratégias disponíveis serão:

**BFS · DFS · UCS · A***

> No próximo notebook, o foco será a implementação da solução no mapa 5×5.
